![hslu_logo.png](img/hslu_logo.png)

## Week 04

<hr style="border:1px solid black">

## Transfer Learning using food and celebrities data
---
---
We use an existing CNN-architecture (VGG16) as baseline and apply transfer learning for a specific problem with limited number of training data

In [ ]:
import torch
from torch.utils.data import Dataset
from torchvision.transforms import v2
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import time
import os
import cv2
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, PrecisionRecallDisplay
import pandas as pd

from utils import plot_img, plot_tiles, plot_error, plot_cost, create_file_frame

def scale_img(image):
    #helper function for scaling image
    return (image - image.min()) / (image.max() - image.min())

### Saving the feature extraction of the VGG16 model

Here we save the feature extraction of the VGG16 model for later use with our model. This code has only to be executed once to create the file `vgg16_classifier.pt`

In [ ]:
#we use this filename below!
vgg_param_file = 'model/vgg16_classifier.pt'

if not os.path.isfile(vgg_param_file):
    #download the model and the weights 
    from torchvision.models import vgg16, VGG16_Weights

    if not os.path.isdir('model'):
        os.mkdir('model')
    
    #prepare the model
    weights = VGG16_Weights.DEFAULT
    vgg16_model = vgg16(weights=weights)

    #take only the feature extraction
    model_to_save = vgg16_model.features
    
    #save to file
    torch.save(model_to_save, vgg_param_file)
    print('file saved')
else:
    print('file already existing')

### Preparation of Data
we have 5 (food) or 8 (celebrities) folders with a set of images for each category; create_file_frame will loop over the folders (train and test) and return a list of pandas DataFrames with files and labels.

Because the sizes of the two image sets are different we set two different default sizes for rescaling at the beginning of the transform chain below.

In [ ]:
problem_type = 0

if problem_type == 0:
    categories = ['apple_braeburn', 'apple_golden_delicious', 'apple_topaz', 'peach', 'pear']
    root_folder = 'food'

    default_size=(300, 400)
else:
    categories = ['angelina jolie', 'catherine deneuve', 'marion cotillard', 'sandra bullock', \
              'brad pitt', 'johnny deep', 'leonardo dicaprio', 'robert de niro']
    root_folder = 'celebrities'

    default_size=(350, 350)

pdFrames, num_categories = create_file_frame(root_folder, categories)

### Define image transformation

For training we apply random rotations (+/-7°), scaling in range [0.95, 1.05], shear (+/-5°) as well as random horizontal flip. Then we center crop the image to 300x300 pixel (original size is 300x400) and resize to 128x128.

For validation and test we only do the center crop an resize.

In [ ]:
#import shortcut (just for here)
from torchvision.transforms import v2

image_size = 128

#transformations including augmentation used for training
train_transform = v2.Compose([
    v2.Resize(default_size, antialias=True),
    v2.RandomAffine(degrees=7, scale=(0.95,1.05), shear=5),
    v2.CenterCrop((300,300)),
    v2.Resize(size=(image_size, image_size), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

#for validation we only do the cropping scaling (size and channels) 
val_transform = v2.Compose([
    v2.Resize(default_size, antialias=True),
    v2.CenterCrop((300,300)),
    v2.Resize(size=[image_size,image_size], antialias=True),  
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


### Define own Datset class

In [ ]:
class MyDataset(Dataset):
    def __init__(self, pd_frame_info, img_dir, transform=None, target_transform=None):
        #can be either file or frames
        if type(pd_frame_info) == pd.core.frame.DataFrame:
            self.img_labels = pd_frame_info
        else:
            self.img_labels = pd.read_csv(pd_frame_info)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = torchvision.io.read_image(img_path, 
                                        mode=torchvision.io.image.ImageReadMode.RGB)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label
        

#### Illustrate usage of DataLoader together with Dataset and transform

In [ ]:
#now create a custom dataset based on the training dataset (we do the same later for the validation and testset)
my_dataset = MyDataset(pdFrames['train'], root_folder, train_transform)

train_loader = torch.utils.data.DataLoader(my_dataset, batch_size=10, shuffle=True)

#setup iterator
data_iterator = iter(train_loader)

#fetch first batch of images (and corresponding labels)
images, labels = next(data_iterator)

#note that the images are a 4-dim tensor; first dimension is image index in batch, second is number of channels (colors)
print(images.shape)
print(labels.shape)
#print first ten labels -> change 'shuffle=True' to 'shuffle=False' to check that order will be always the same
print(labels[:10])


In [ ]:
images, labels = next(data_iterator)

#we have to reshape (we are not prepared to show channels)
plot_tiles(scale_img(images), cols=5)

#### Setup model architectures

We use feature extraction from vgg16 and add one dense layer. With input image resolution of 3x128x128 we end up with 512x4x4 features maps after 5 pooling layers with stride 2. This sums up to 8192 features as input to our dense layer. We make the last two convolutional layers and the dense layers trainable.

In [ ]:
def ceate_transfer_learn_model(param_file, print_model=False, print_freeze= False):
    #load the vgg16 feature extraction part from file 
    my_model = torch.load(param_file)

    #fix all but last two conv layers
    for param in my_model[:-5].parameters():
        param.requires_grad = False

    #add own layers (will not be freezed)
    ind = len(my_model)
    my_model.add_module(str(ind), torch.nn.Flatten())
    my_model.add_module(str(ind+1), torch.nn.Linear(in_features=8192, out_features=256))
    my_model.add_module(str(ind+2), torch.nn.ReLU())
    my_model.add_module(str(ind+3), torch.nn.Dropout(p=0.5, inplace=False))
    my_model.add_module(str(ind+4), torch.nn.Linear(in_features=256, out_features=num_categories))
      
    if print_model:
        print(my_model)
    
    if print_freeze:
        for ind, mod in enumerate(my_model):
            print(f'({ind})', mod)
            for param in mod.parameters():
                print(param.requires_grad)

    return my_model

### Only to investigate the model

This cell is only to investigate the model used for tansfer learning. Set either `print_model` or `print_freeze` to visualise model or trainable status of the layers respectively.

In [ ]:
my_model = ceate_transfer_learn_model(vgg_param_file, print_model=False, print_freeze=False)

### Class NeuralNetwork

This class constructs the network. Cost function is CE. The method $propagate()$ returns the prediction $$ \hat{y}^{(i)}=h_\theta(\mathbf{x}^{(i)}) $$ on the input data and $back\_propagate()$ determines the gradients of the cost function with respect to the parameters (weights and bias for all layers) $$ \nabla_{\mathbf{\theta}} J(\mathbf{\theta}) $$
The method $gradient\_descend()$ finally does the correction of the parameters with a step in the negative gradient direction, weighted with the learning rate $$\alpha$$ for all layers.

In [ ]:
class NeuralNetwork:
    """
    Neural network class handling the training and inference
    """
    def __init__(self, cnn_model):
        """
        constructor

        Arguments:
        cnn_model -- the model to use
        
        """
        self.model = cnn_model
            
        self.cost_fn = torch.nn.CrossEntropyLoss(reduction='mean')

        #used to save results
        self.result_data = torch.tensor([])
        
        #we keep a global step counter, thus that optimise can be called 
        #several times with different settings
        self.epoch_counter = 0 
        
    def propagate(self, x):
        """
        calculates the function estimation based on current parameters
        """    
        y_pred = self.model(x)

        return y_pred
           
     
    def back_propagate(self, cost):
        """
        calculates the backpropagation results based on expected output y
        this function must be performed AFTER the corresponding propagte step
        """    
        #set gradient values to zero
        self.model.zero_grad()
        
        cost.backward()
 

    def cost_funct(self, y_pred, y):
        """
        calculates the MSE loss function
        """
        cost = self.cost_fn(y_pred, y)
        
        return cost
    
         
    def gradient_descend(self, alpha):
        """
        does the gradient descend based on results from last back_prop step with learning rate alpha
        """
        with torch.no_grad():
            for param in self.model.parameters():
                if param.requires_grad == True:
                    param -= alpha * param.grad
            
         
    def calc_error(self, y_pred, y):
        """
        get error information
        """
        m = y.shape[0]

        y_pred_argmax = torch.argmax(y_pred, dim=1)
        error = torch.sum(y != y_pred_argmax) / m

        return error

    
    def append_result(self, train_res):
        """
        append cost and error data to output array
        """
        #this takes quite a long time (transform is applied to all images) but is only executed once 
        #then the images are available for quick execution of propagation step
        if self.epoch_counter == 0: 
            # dataloaders (we use original set (training/test_data); own data has to realize the abstract class representing 'Dataset'
            valid_loader = torch.utils.data.DataLoader(self.data['valid'], batch_size=len(self.data['valid']), shuffle=False)
            valid_iterator = iter(valid_loader)
            self.valid_images, self.valid_labels = next(valid_iterator) 
            
        # determine cost and error functions for train and validation data
        y_pred_val = self.propagate(self.valid_images)
        cost_val = self.cost_funct(y_pred_val, self.valid_labels)
        error_val = self.calc_error(y_pred_val, self.valid_labels)

        res_data = torch.tensor([[train_res[0] if train_res.size else cost_val, 
                                  train_res[1] if train_res.size else error_val,
                                  cost_val, error_val]])
        #save model with lowest cost value
        if self.result_data.numel() and (cost_val < self.result_data[:,2].min()):
            torch.save(self.model, f'model/best_model.pt') 
        
        self.result_data = torch.cat((self.result_data, res_data), 0)

        #increase epoch counter here (used for plot routines below)
        self.epoch_counter += 1 
        
        return res_data

        
    def optimise(self, data, epochs, alpha, batch_size=0, debug=0):
        """
        performs epochs number of gradient descend steps and appends result to output array

        Arguments:
        data -- dictionary with NORMALISED data
        epochs -- number of epochs
        alpha -- learning rate
        batch_size -- size of batches (1 = SGD, 1 < .. < n = mini-batch)
        debug -- integer value; get info on gradient descend step every debug-step (0 -> no output)
        """
        #access to data from other methods
        self.data = data

        # dataloader for training image
        train_loader = torch.utils.data.DataLoader(data['train'], batch_size=batch_size, 
                                                   shuffle=True)
        
        # save results before 1st step
        if self.epoch_counter == 0:
            self.model.eval()
            res_data = self.append_result(np.array([]))

        self.model.train()
        for i0 in range(0, epochs):    
            #measure time for one epoch
            start=time.time()
            #setup loop over all batchs
            data_iterator = iter(train_loader)
            res_epoch = np.zeros(2)
            for batch_iter in data_iterator:
                #do prediction
                y_pred = self.propagate(batch_iter[0])
                #determine the loss 
                cost = self.cost_funct(y_pred, batch_iter[1])
                #determine the error
                self.back_propagate(cost)
                #do the correction step
                self.gradient_descend(alpha)
                #add up results for each batch
                res_epoch[0] += cost.detach().item()
                res_epoch[1] += self.calc_error(y_pred, batch_iter[1])

            #save result (for train use average over batches)
            self.model.eval()
            res_data = self.append_result(res_epoch/data_iterator.__len__())

            #end of time measurement
            end=time.time()
            
            if debug and np.mod(i0, debug) == 0:
                print('result after %d epochs (dt=%1.2f s), train: cost %.5f, error %.5f ; validation: cost %.5f, error %.5f'
                    % (self.epoch_counter-1, end-start, res_data[0, 0].item(), res_data[0, 1].item(), \
                                                                res_data[0, 2].item(), res_data[0, 3].item()))

        if debug:
            print('result after %d epochs, train: cost %.5f, error %.5f ; validation: cost %.5f, error %.5f'
                  % (self.epoch_counter-1, res_data[0, 0].item(), res_data[0, 1].item(), \
                                                                res_data[0, 2].item(), res_data[0, 3].item()))
                        
            

### Sample execution of Neural Network

The cell below shows how to use the class NeuralNetwork and how to perform the optimisation. The training and test data is given as dictionary in the call to the method $optimise()$. This method can be called several times in a row with different arguments.

In [ ]:
#always read the model form scratch
my_model = ceate_transfer_learn_model(vgg_param_file)

num_samples = len(pdFrames['train'])
indices = np.random.permutation(num_samples)

validation_size = 0.2
valid_ind = int(num_samples*(1-validation_size))

#create custom training and validation data set
train_dataset = MyDataset(pdFrames['train'].iloc[indices[:valid_ind],:], root_folder, train_transform)
valid_dataset = MyDataset(pdFrames['train'].iloc[indices[valid_ind:],:], root_folder, val_transform)


#data is arranged as dictionary with quick access through respective keys
data = {'train' : train_dataset, 'valid' : valid_dataset}

#my_model as parameter
NNet = NeuralNetwork(my_model)

#choose the hyperparameters you want to use for training
epochs = 15
batchsize = 16
learning_rate = 0.005
NNet.optimise(data, epochs, learning_rate, batchsize, debug=3)


plot_error(NNet, y_range=[0.001, 1])
plot_cost(NNet, y_range=[0.01, 1])


In [ ]:
#analyse false classified training on test images
#also prepare the test dataset
test_dataset = MyDataset(pdFrames['test'], root_folder, val_transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)
test_iterator = iter(test_loader)
test_images, test_labels = next(test_iterator)

y_pred = torch.argmax(NNet.propagate(test_images), axis=1)
num_false_class = torch.sum(y_pred != test_labels)

print(f'test error rate: {num_false_class} out of {y_pred.shape[0]}, {100*num_false_class/y_pred.shape[0]:.1f}%')


In [ ]:
#test best model saved
best_model = torch.load(f'model/best_model.pt') 

y_pred = torch.argmax(best_model(test_images), axis=1)
num_false_class = torch.sum(y_pred != test_labels)

print(f'test error rate: {num_false_class} out of {y_pred.shape[0]}, {100*num_false_class/y_pred.shape[0]:.1f}%')

In [ ]:
#plot all false classified images 
indices_false = torch.where(y_pred != test_labels)[0]

for ind, img_ind in enumerate(indices_false):
    title = f'#{ind+1}: category is {categories[test_labels[img_ind].item()]}, prediction is {categories[y_pred[img_ind].item()]}'
    plot_img(torch.movedim(scale_img(test_images[img_ind]), [0],[2]), figure_size = [2,2], fig_title=title)

In [ ]:
confMat = confusion_matrix(test_labels, y_pred,labels=np.arange(num_categories))
#print(confMat)
#print(confMat/np.sum(confMat,axis=1))

disp=ConfusionMatrixDisplay(confMat,display_labels=categories)
disp.plot(xticks_rotation=45)
plt.show()